# Model D — Model B + HVI-Inspired Color Correction (Colab T4)

**Research goal:** Preserve the already trained **Model B** as the baseline. Train **only** an HVI-inspired, intensity-guided residual color correction head on Model B outputs. Compare Model B versus Model D using **PSNR, SSIM and matched visual images**.

**Run sections from top to bottom.** Each section ends with a distinct check and clear output. Do not click *Run all* before Section 4 succeeds.

- **Model B:** identical frozen NAF-inspired model/loading pattern from your provided Option A notebook (original verified v1 commit + IG-MSA replacement).
- **Training split:** original 620 Train / 69 held-out validation, loaded from existing `split.json`; **official LOLv2 Test** is evaluated only at the end.
- **No full CIDNet:** HVI color-space idea + a compact CNN that predicts color-only HV residuals; intensity stays unchanged.
- **Expected result:** `best_model_d_head.pth`, Model B/Model D PNG images, `validation.csv`, `test_summary.csv`, `test_per_image.csv`, side-by-side PNGs, training curve, ZIP report.

**Model B checkpoint stays read-only. Model D result files go to a new folder.**

## Section 1 — GPU and Drive setup
**Expected:** `GPU: Tesla T4` (or other CUDA GPU), Drive mounted, and package imports successful.

In [ ]:
import os, sys, subprocess, json, csv, random, time, hashlib, math, gc, shutil
from pathlib import Path
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
import torch, numpy as np
from PIL import Image
print('Python:', sys.version.split()[0], '| Torch:', torch.__version__)
assert torch.cuda.is_available(), 'Select Colab Runtime > Change runtime type > T4 GPU'
DEVICE=torch.device('cuda')
print('GPU:', torch.cuda.get_device_name(0), '| VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory/1e9,2))
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
import matplotlib.pyplot as plt
from torchvision.transforms import functional as TF
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
print('PASS: setup and Drive mount')

## Section 2 — Central paths and hyperparameters
**Only edit this cell if your Drive layout changes.** All other cells use these variables.

In [ ]:
# ----- VERIFIED paths from Model B Option A notebook -----
DRIVE=Path('/content/drive/MyDrive')
DATA_ROOT=DRIVE/'datasets/LOLv2/Real_captured'
SPLIT_FILE=DRIVE/'RetinexFormer_NAF_Screening/model_B_v1_full_data_1000steps/split.json'
MODEL_B_CKPT=DRIVE/'RetinexFormer_NAF_Screening/model_B_optionA_batch8_patch128_lr1e-5_from_batch8best/low_lr_1e-5_1000steps/best.pth'
# Separate new directory: NEVER overwrite Model B work.
RUN_ROOT=DRIVE/'RetinexFormer_NAF_Screening/model_D_HVI_from_ModelB_OptionA_v1'
CACHE_ROOT=Path('/content/model_d_cache')  # Colab local disk; regenerated after runtime reset.

# ----- Initial experimental settings -----
SEED=42
PATCH=128
BATCH=8
EPOCHS=30
LR=1e-4
WEIGHT_DECAY=0.0
HEAD_WIDTH=32
HVI_K=1.0       # Fixed intensity-collapse parameter for first controlled study
MAX_DELTA=0.12 # Small bounded color-only correction
LAMBDA_HV=0.10
LAMBDA_RES=0.01
MODEL_B_AMP=True
CACHE_TRAIN=True
CACHE_VAL=True
CACHE_TEST=True
SAVE_EVERY_EPOCH=True
ARCH=dict(in_channels=3,out_channels=3,n_feat=40,stage=1,num_blocks=[1,2,2])
ORIGINAL_COMMIT='ad33e6bf5b338bc36e4a14102781e78a03397b3b'

for tag,path in [('Train low',DATA_ROOT/'Train/Low'),('Train GT',DATA_ROOT/'Train/Normal'),
                 ('Test low',DATA_ROOT/'Test/Low'),('Test GT',DATA_ROOT/'Test/Normal'),
                 ('620/69 split',SPLIT_FILE),('Model B checkpoint',MODEL_B_CKPT)]:
    ok=path.is_dir() if tag in ('Train low','Train GT','Test low','Test GT') else path.is_file()
    print(('OK     ' if ok else 'MISSING'),tag,':',path)
    assert ok, f'Fix the path of {tag} before continuing: {path}'
RUN_ROOT.mkdir(parents=True,exist_ok=True)
CACHE_ROOT.mkdir(parents=True,exist_ok=True)
print('PASS: Required source paths exist. Model D destination:',RUN_ROOT)

## Section 3 — Fetch the exact verified Model B architecture
**Expected:** pinned commit, model instantiated with NAF mixer and `edge_guidance=True`. Same implementation method as the uploaded Model B Option A notebook.

In [ ]:
REPO=Path('/content/myRetinexformer_modeld')
if not (REPO/'.git').exists():
    subprocess.run(['git','clone','--branch','v1','--single-branch',
                    'https://github.com/Silly-Gooses/myRetinexformer.git',str(REPO)],check=True)
subprocess.run(['git','fetch','origin','v1'],cwd=REPO,check=True)
subprocess.run(['git','checkout','--detach',ORIGINAL_COMMIT],cwd=REPO,check=True)
COMMIT=subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()
assert COMMIT==ORIGINAL_COMMIT, f'Unexpected repo commit: {COMMIT}'
sys.path.insert(0,str(REPO))
from RetinexFormer_arch import RetinexFormer, IG_MSA
from analysis_utils import find_paired_images, image_metrics, tensor_to_uint8
print('PASS: pinned verified Model B source at', COMMIT)

In [ ]:
# Exact NAF-inspired replacement from the uploaded Model B Option A notebook.
class NAFInspiredMixer(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.expand=nn.Conv2d(dim,2*dim,1)
        self.dw=nn.Conv2d(2*dim,2*dim,3,padding=1,groups=2*dim)
        self.sca=nn.Conv2d(dim,dim,1)
        self.project=nn.Conv2d(dim,dim,1)
        nn.init.zeros_(self.project.weight);nn.init.zeros_(self.project.bias)
    def forward(self,x_in,illu_fea_trans=None):
        x=x_in.permute(0,3,1,2).contiguous()
        x=self.dw(self.expand(x))
        a,b=x.chunk(2,dim=1)
        x=a*b
        x=x*self.sca(F.adaptive_avg_pool2d(x,1))
        return self.project(x).permute(0,2,3,1).contiguous()

def build_model_b():
    model=RetinexFormer(**ARCH,edge_guidance=True)
    replaced=[]
    for name,mod in list(model.named_modules()):
        if isinstance(mod,IG_MSA):
            parent_path,_,leaf=name.rpartition('.')
            parent=model.get_submodule(parent_path)
            if isinstance(parent,(nn.ModuleList,nn.Sequential)):
                parent[int(leaf)]=NAFInspiredMixer(mod.dim)
            else:
                setattr(parent,leaf,NAFInspiredMixer(mod.dim))
            replaced.append(name)
    assert replaced and not any(isinstance(m,IG_MSA) for m in model.modules())
    return model,replaced

model_b,replaced=build_model_b()
print('Model B IG_MSA replaced:',len(replaced),replaced)
print('Model B parameters:',sum(p.numel() for p in model_b.parameters()))
assert len(replaced)>0
print('PASS: Model B architecture built')

## Section 4 — Check checkpoint and reproduce Model B inference
**Critical gate:** `load_state_dict(..., strict=True)` must pass before creating Model D data. Checkpoint is loaded *read-only*. The architecture is not retrained.

In [ ]:
ckpt=torch.load(MODEL_B_CKPT,map_location='cpu',weights_only=False)  # TRUSTED self-generated checkpoint only
assert isinstance(ckpt,dict) and 'model' in ckpt, f'Unexpected Model B checkpoint keys: {list(ckpt)[:15]}'
state=ckpt['model']
model_b.load_state_dict(state,strict=True)
if 'config' in ckpt and 'arch' in ckpt['config']:
    assert ckpt['config']['arch']==ARCH,'Saved checkpoint architecture does not match.'
model_b=model_b.to(DEVICE).eval()
for p in model_b.parameters():p.requires_grad_(False)
MODEL_B_SHA256=hashlib.sha256(MODEL_B_CKPT.read_bytes()).hexdigest()
print('Model B checkpoint:',MODEL_B_CKPT)
print('SHA256:',MODEL_B_SHA256)
print('Saved best PSNR (metadata only):',ckpt.get('best_psnr','not stored'))
print('Saved step:',ckpt.get('local_step','not stored'))
print('Frozen weights:',all(not p.requires_grad for p in model_b.parameters()))
print('PASS: exact Model B strict-load')
del ckpt,state

@torch.inference_mode()
def model_b_predict(x):
    h,w=x.shape[-2:]
    xpad=F.pad(x,(0,(-w)%4,0,(-h)%4),mode='replicate')
    with torch.autocast('cuda',enabled=MODEL_B_AMP):
        out=model_b(xpad)
    if isinstance(out,(tuple,list)):out=out[0]
    return out[:,:,:h,:w].float().clamp(0,1)

x=torch.rand(1,3,128,128,device=DEVICE)
y=model_b_predict(x)
assert y.shape==x.shape and torch.isfinite(y).all()
print('PASS: Model B forward check | shape:',tuple(y.shape))

## Section 5 — Read the original dataset split (no leakage)
**Expected:** 620 training and 69 validation images, plus the independent official Test folder. Do not create another random split.

In [ ]:
def get_pairs(low_dir,gt_dir):
    return find_paired_images(low_dir,gt_dir)
all_train=get_pairs(DATA_ROOT/'Train/Low',DATA_ROOT/'Train/Normal')
by_name={p[0]:p for p in all_train}
original=json.loads(SPLIT_FILE.read_text())
train_names=list(original['train']);val_names=list(original['val'])
assert len(train_names)==620 and len(val_names)==69,(len(train_names),len(val_names))
assert not(set(train_names)&set(val_names)),'Train/Val leakage'
assert set(train_names)|set(val_names)==set(by_name),'Split does not match dataset filenames'
train_pairs=[by_name[k] for k in train_names]
val_pairs=[by_name[k] for k in val_names]
test_pairs=get_pairs(DATA_ROOT/'Test/Low',DATA_ROOT/'Test/Normal')
assert len(test_pairs)>0,'No official test image pairs found'
# Official Test filenames can overlap Train filenames; they reside in separate official directories.
print('Train:',len(train_pairs),'| Validation:',len(val_pairs),'| Official Test:',len(test_pairs))
print('Train sample:',train_pairs[0][0],'| Val sample:',val_pairs[0][0],'| Test sample:',test_pairs[0][0])
print('PASS: original 620/69 split and official Test are separate')

## Section 6 — Generate Model B output images (first time only)
This step uses the **frozen** Model B. It saves enhanced PNG images in Colab local disk so head training is fast. It does not change the Model B checkpoint. **After Colab runtime reset, rerun this step** (cache lives in `/content`).

In [ ]:
def rgb_tensor(path):
    with Image.open(path) as im:arr=np.asarray(im.convert('RGB')).copy()
    return torch.from_numpy(arr).permute(2,0,1).float()/255.0

def generate_cache(split_name,pairs):
    target=CACHE_ROOT/split_name
    (target/'b').mkdir(parents=True,exist_ok=True)
    (target/'gt').mkdir(parents=True,exist_ok=True)
    rows=[]
    tic=time.time()
    for ix,(name,low_path,gt_path) in enumerate(pairs,1):
        bfile=target/'b'/name;gtfile=target/'gt'/name
        if not bfile.exists():
            low=rgb_tensor(low_path).unsqueeze(0).to(DEVICE)
            pred=model_b_predict(low)
            Image.fromarray(tensor_to_uint8(pred[0])).save(bfile)
            del low,pred
        if not gtfile.exists():
            shutil.copy2(gt_path,gtfile)
        with Image.open(bfile) as im:s1=im.size
        with Image.open(gtfile) as im:s2=im.size
        assert s1==s2, f'Model B/GT size mismatch: {name}: {s1} vs {s2}'
        rows.append((name,str(bfile),str(gtfile)))
        if ix==1 or ix%100==0 or ix==len(pairs):
            print(f'{split_name}: {ix}/{len(pairs)} | {time.time()-tic:.1f}s elapsed')
    assert len(rows)==len(pairs)
    return rows

# Run each independently so you can identify failures by split.
train_cached=generate_cache('train',train_pairs) if CACHE_TRAIN else []
print('PASS: Train cached',len(train_cached))

In [ ]:
val_cached=generate_cache('val',val_pairs) if CACHE_VAL else []
print('PASS: Validation cached',len(val_cached))

In [ ]:
test_cached=generate_cache('test',test_pairs) if CACHE_TEST else []
print('PASS: Official Test cached',len(test_cached))

## Section 7 — Sanity check Model B baseline PSNR and SSIM
This uses **the exact `analysis_utils.image_metrics` function** from Model B, on Model B PNGs and Ground Truth. Compare validation PSNR to approximately **21.0366 dB**. A significant mismatch is a stop sign: check checkpoint, image conversion and dataset pairing before training.

In [ ]:
def baseline_metrics(cached_pairs):
    results=[]
    for name,bpath,gtpath in cached_pairs:
        with Image.open(bpath) as im:b=np.asarray(im.convert('RGB'))
        with Image.open(gtpath) as im:g=np.asarray(im.convert('RGB'))
        psnr,ssim=image_metrics(b,g)
        results.append(dict(name=name,psnr=float(psnr),ssim=float(ssim)))
    return pd.DataFrame(results)

val_baseline=baseline_metrics(val_cached)
print('Validation B0 PSNR:',round(val_baseline.psnr.mean(),5),'SSIM:',round(val_baseline.ssim.mean(),6))
print('Reference: PSNR ~21.03665, SSIM ~0.735456 (69 images)')
assert len(val_baseline)==69
# PNG quantization or implementation differences may create a small discrepancy; fail on large mismatches.
if abs(val_baseline.psnr.mean()-21.036645)>0.25:
    raise RuntimeError('Model B validation differs >0.25dB from verified run. Stop and investigate.')
print('PASS: baseline validation sanity check')

## Section 8 — HVI-inspired color conversion and round-trip test
This is **not** the full original CIDNet. We use polarized hue (`cos`, `sin`), saturation, and the paper-inspired intensity collapse `C_k`; color residual head predicts only HV offsets. The intensity channel remains unchanged.

**Test:** RGB → HVI → RGB should reconstruct a random input with negligible error. `k=1` is intentionally fixed for the first ablation.

In [ ]:
# HVI mapping adapted for color-only residual correction. Float32 required for stable angles.
def rgb_to_hvi(rgb,k=HVI_K):
    rgb=rgb.float().clamp(0,1)
    r,g,b=rgb[:,0:1],rgb[:,1:2],rgb[:,2:3]
    hi=rgb.max(1,keepdim=True).values
    lo=rgb.min(1,keepdim=True).values
    delta=hi-lo
    sat=torch.where(hi>1e-8,delta/hi.clamp_min(1e-8),torch.zeros_like(hi))
    # Hue in [0, 1); mask excludes grayscale where the value does not matter.
    dr=(g-b)/delta.clamp_min(1e-8)
    dg=(b-r)/delta.clamp_min(1e-8)+2
    db=(r-g)/delta.clamp_min(1e-8)+4
    hue=torch.where(hi==r,dr,torch.where(hi==g,dg,db))
    hue=torch.remainder(hue/6,1.0)
    theta=2*math.pi*hue
    ck=(torch.sin(math.pi*hi/2).clamp_min(1e-8))**(1/k)
    hv=torch.cat([ck*sat*torch.cos(theta),ck*sat*torch.sin(theta)],dim=1)
    return hv,hi

def hsv_to_rgb(hue,sat,val):
    hue=torch.remainder(hue,1.0)
    sat=sat.clamp(0,1);val=val.clamp(0,1)
    h6=hue*6
    c=val*sat
    x=c*(1-torch.abs(torch.remainder(h6,2)-1))
    z=torch.zeros_like(c)
    i=torch.floor(h6).long().clamp(0,5)
    R=torch.where(i==0,c,torch.where(i==1,x,torch.where(i==2,z,torch.where(i==3,z,torch.where(i==4,x,c)))))
    G=torch.where(i==0,x,torch.where(i==1,c,torch.where(i==2,c,torch.where(i==3,x,z))))
    B=torch.where(i==0,z,torch.where(i==1,z,torch.where(i==2,x,torch.where(i==3,c,torch.where(i==4,c,x)))))
    return torch.cat([R+val-c,G+val-c,B+val-c],1).clamp(0,1)

def hvi_to_rgb(hv,intensity,k=HVI_K):
    hv=hv.float();intensity=intensity.float().clamp(0,1)
    ck=torch.sin(math.pi*intensity/2).clamp_min(1e-8)**(1/k)
    hue=torch.remainder(torch.atan2(hv[:,1:2],hv[:,0:1])/(2*math.pi),1.0)
    sat=torch.linalg.vector_norm(hv,dim=1,keepdim=True)/ck.clamp_min(1e-8)
    return hsv_to_rgb(hue,sat,intensity)

with torch.no_grad():
    x=torch.rand(2,3,64,64,device=DEVICE)
    hv,ii=rgb_to_hvi(x);xr=hvi_to_rgb(hv,ii)
    err=(x-xr).abs().max().item()
    print('HVI RGB round-trip max error:',err)
    assert err<2e-4,f'HVI round-trip incorrect: {err}'
print('PASS: HVI conversion / inverse verified')

## Section 9 — Build the trainable Model D color-only head
The head sees `[HV, intensity]` (3 channels), predicts only a small **two-channel HV residual** and does **not** predict brightness changes. Final projection starts at zero ⇒ initially Model D≈Model B.

In [ ]:
class HVIColorHead(nn.Module):
    def __init__(self,width=HEAD_WIDTH,max_delta=MAX_DELTA):
        super().__init__()
        self.net=nn.Sequential(
            nn.Conv2d(3,width,3,padding=1),nn.ReLU(inplace=True),
            nn.Conv2d(width,width,3,padding=1),nn.ReLU(inplace=True),
            nn.Conv2d(width,width,3,padding=1),nn.ReLU(inplace=True),
            nn.Conv2d(width,2,3,padding=1))
        self.max_delta=max_delta
        nn.init.zeros_(self.net[-1].weight)
        nn.init.zeros_(self.net[-1].bias)
    def forward(self,rgb):
        hv,ii=rgb_to_hvi(rgb)
        delta=self.max_delta*torch.tanh(self.net(torch.cat([hv,ii],1)))
        hv_corr=hv+delta
        out=hvi_to_rgb(hv_corr,ii)
        return out,delta,hv_corr

head=HVIColorHead().to(DEVICE)
print('Trainable Model D head parameters:',sum(p.numel() for p in head.parameters()))
with torch.no_grad():
    demo=torch.rand(1,3,128,128,device=DEVICE)
    out,delta,_=head(demo)
    print('Before training: maximum output RGB difference:',(out-demo).abs().max().item())
    assert (out-demo).abs().max().item()<2e-4
print('PASS: model starts as identity correction')

## Section 10 — Model D head dataset (paired random crops)
Random crop is applied **identically** to enhanced Model B output and corresponding normal-light Ground Truth. Only the head receives gradients.

In [ ]:
class ColorPairs(Dataset):
    def __init__(self,cache,patch=PATCH,augment=False):
        self.cache=cache;self.patch=patch;self.augment=augment
    def __len__(self):return len(self.cache)
    def __getitem__(self,idx):
        name,bpath,gtpath=self.cache[idx]
        with Image.open(bpath) as im:aa=im.convert('RGB')
        with Image.open(gtpath) as im:bb=im.convert('RGB')
        assert aa.size==bb.size,name
        w,h=aa.size;p=self.patch
        assert h>=p and w>=p,f'Image too small: {name}'
        top=random.randrange(h-p+1);left=random.randrange(w-p+1)
        aa=TF.crop(aa,top,left,p,p);bb=TF.crop(bb,top,left,p,p)
        if self.augment:
            if random.random()<0.5:aa=TF.hflip(aa);bb=TF.hflip(bb)
            if random.random()<0.5:aa=TF.vflip(aa);bb=TF.vflip(bb)
            if random.random()<0.5:aa=TF.rotate(aa,90);bb=TF.rotate(bb,90)
        return TF.to_tensor(aa),TF.to_tensor(bb)

def seed_all(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed);torch.cuda.manual_seed_all(seed)
seed_all(SEED)
train_loader=DataLoader(ColorPairs(train_cached,augment=True),batch_size=BATCH,shuffle=True,
                        num_workers=2,pin_memory=True,drop_last=False,persistent_workers=True)
print('Training pairs:',len(train_cached),'| Batches/epoch:',len(train_loader))
print('PASS: head dataloader')

## Section 11 — Validation functions and training settings
**Selection metric:** mean validation PSNR on the original 69 held-out images. The official Test set is never used for checkpoint selection.

In [ ]:
@torch.inference_mode()
def evaluate_head(head,cached,return_images=False):
    head.eval();rows=[];outputs=[]
    for name,bpath,gtpath in cached:
        b=rgb_tensor(bpath).unsqueeze(0).to(DEVICE)
        g=rgb_tensor(gtpath).unsqueeze(0).to(DEVICE)
        with torch.autocast('cuda',enabled=False):
            pred,_,_=head(b)
        pred_np=tensor_to_uint8(pred[0]);gt_np=tensor_to_uint8(g[0])
        psnr,ssim=image_metrics(pred_np,gt_np)
        rows.append(dict(name=name,psnr=float(psnr),ssim=float(ssim)))
        if return_images:outputs.append((name,pred_np))
    out=pd.DataFrame(rows)
    return out,outputs

opt=torch.optim.AdamW(head.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
HISTORY_PATH=RUN_ROOT/'training_history.csv'
BEST_HEAD=RUN_ROOT/'best_model_d_head.pth'
LAST_HEAD=RUN_ROOT/'last_model_d_head.pth'
VAL_CSV=RUN_ROOT/'validation.csv'
CONFIG=dict(model_b_checkpoint=str(MODEL_B_CKPT),model_b_sha256=MODEL_B_SHA256,
            repo_commit=COMMIT,arch=ARCH,seed=SEED,patch=PATCH,batch=BATCH,epochs=EPOCHS,
            lr=LR,head_width=HEAD_WIDTH,hvi_k=HVI_K,max_delta=MAX_DELTA,
            lambda_hv=LAMBDA_HV,lambda_res=LAMBDA_RES,
            train_names=train_names,val_names=val_names,test_count=len(test_pairs))
CONFIG_PATH=RUN_ROOT/'run_config.json'
if CONFIG_PATH.exists():
    assert json.loads(CONFIG_PATH.read_text())==CONFIG, 'Existing Model D run has different config. Change RUN_ROOT.'
else: CONFIG_PATH.write_text(json.dumps(CONFIG,indent=2))
print('Initial Model B validation PSNR:',val_baseline.psnr.mean())
print('PASS: evaluation setup')

## Section 12 — Train only the Model D HVI color head
**Run this cell for the main experiment.** Checkpoint is saved every epoch; interrupted training can resume. `Model B` is always frozen. After every epoch, validation PSNR/SSIM and training loss are printed.

In [ ]:
def save_head(path,epoch,best_score):
    temp=path.with_suffix('.tmp')
    torch.save(dict(epoch=epoch,best_val_psnr=best_score,head=head.state_dict(),
                    optimizer=opt.state_dict(),config=CONFIG),temp)
    os.replace(temp,path)

def run_training():
    seed_all(SEED)
    start=0;best=-float('inf');records=[]
    if LAST_HEAD.exists():
        state=torch.load(LAST_HEAD,map_location='cpu',weights_only=False) # trusted self-generated
        assert state['config']==CONFIG,'Resume config mismatch'
        head.load_state_dict(state['head'],strict=True)
        opt.load_state_dict(state['optimizer'])
        start=int(state['epoch']);best=float(state['best_val_psnr'])
        print('Resuming from completed epoch',start)
    if HISTORY_PATH.exists():records=pd.read_csv(HISTORY_PATH).to_dict('records')
    if start==EPOCHS:
        print('Training already complete. Reusing saved best checkpoint.')
        return
    for epoch in range(start+1,EPOCHS+1):
        head.train();losses=[];t0=time.time()
        for b,g in train_loader:
            b=b.to(DEVICE,non_blocking=True);g=g.to(DEVICE,non_blocking=True)
            opt.zero_grad(set_to_none=True)
            # All head/HVI arithmetic in float32 for consistent inverse conversion.
            output,delta,hv_corrected=head(b)
            hv_gt,_=rgb_to_hvi(g)
            loss_rgb=F.l1_loss(output,g)
            loss_hv=F.l1_loss(hv_corrected,hv_gt)
            loss_res=delta.abs().mean()
            loss=loss_rgb+LAMBDA_HV*loss_hv+LAMBDA_RES*loss_res
            if not torch.isfinite(loss):raise FloatingPointError(f'Non-finite loss at epoch {epoch}')
            loss.backward();opt.step()
            losses.append(loss.item())
        vdf,_=evaluate_head(head,val_cached)
        score=float(vdf.psnr.mean());ssim=float(vdf.ssim.mean())
        row=dict(epoch=epoch,train_loss=float(np.mean(losses)),val_psnr=score,
                 val_ssim=ssim,elapsed_sec=round(time.time()-t0,2))
        records.append(row)
        pd.DataFrame(records).to_csv(HISTORY_PATH,index=False)
        pd.DataFrame(records)[['epoch','val_psnr','val_ssim']].to_csv(VAL_CSV,index=False)
        if score>best:
            best=score;save_head(BEST_HEAD,epoch,best);label='NEW BEST'
        else:label=''
        save_head(LAST_HEAD,epoch,best)
        print(f"Epoch {epoch:02d}/{EPOCHS}: loss={row['train_loss']:.5f}  val PSNR={score:.4f}  SSIM={ssim:.5f}  best={best:.4f}  {label}",flush=True)
    print('PASS: Model D training completed. Best head:',BEST_HEAD)

run_training()

## Section 13 — Validate final best head against Model B (before Test)
Expected: a comparison on the exact **same 69 validation images**. If Model D performs worse, the result is still important and should be reported.

In [ ]:
assert BEST_HEAD.exists(),'Train Section 12 first.'
best=torch.load(BEST_HEAD,map_location='cpu',weights_only=False) # trusted self-generated
assert best['config']==CONFIG
head.load_state_dict(best['head'],strict=True)
head.eval()
val_d,_=evaluate_head(head,val_cached)
val_table=pd.DataFrame([
    dict(model='Model B (baseline)',psnr=val_baseline.psnr.mean(),ssim=val_baseline.ssim.mean()),
    dict(model='Model D (B + HVI head)',psnr=val_d.psnr.mean(),ssim=val_d.ssim.mean())])
print(val_table.to_string(index=False,float_format=lambda v:f'{v:.5f}'))
print('D - B delta PSNR:',round(float(val_d.psnr.mean()-val_baseline.psnr.mean()),5),'dB')
print('Best Model D epoch:',best['epoch'])
val_table.to_csv(RUN_ROOT/'validation_comparison.csv',index=False)
print('PASS: Validation comparison generated')

## Section 14 — FINAL unbiased test: Model B vs Model D
The official LOLv2 Test set is evaluated **once after training**. Outputs are saved to separate folders; test PSNR and SSIM are recomputed for both models with the same metric function.

In [ ]:
test_b=baseline_metrics(test_cached)
test_d,images_d=evaluate_head(head,test_cached,return_images=True)
assert list(test_b.name)==list(test_d.name),'Model B/Model D test pairing mismatch'
OUTPUTS_B=RUN_ROOT/'test_outputs/ModelB'
OUTPUTS_D=RUN_ROOT/'test_outputs/ModelD'
OUTPUTS_B.mkdir(parents=True,exist_ok=True)
OUTPUTS_D.mkdir(parents=True,exist_ok=True)
for name,bpath,_ in test_cached:
    shutil.copy2(bpath,OUTPUTS_B/name)
for name,arr in images_d:
    Image.fromarray(arr).save(OUTPUTS_D/name)
comparison=test_b.rename(columns={'psnr':'psnr_B','ssim':'ssim_B'}).merge(
    test_d.rename(columns={'psnr':'psnr_D','ssim':'ssim_D'}),on='name',validate='one_to_one')
comparison['delta_psnr']=comparison.psnr_D-comparison.psnr_B
comparison['delta_ssim']=comparison.ssim_D-comparison.ssim_B
comparison.to_csv(RUN_ROOT/'test_per_image.csv',index=False)
summary=pd.DataFrame([
    dict(model='Model B',count=len(test_b),psnr=test_b.psnr.mean(),ssim=test_b.ssim.mean()),
    dict(model='Model D',count=len(test_d),psnr=test_d.psnr.mean(),ssim=test_d.ssim.mean())])
summary.to_csv(RUN_ROOT/'test_summary.csv',index=False)
print(summary.to_string(index=False,float_format=lambda v:f'{v:.5f}'))
print('PSNR delta (D-B):',round(comparison.delta_psnr.mean(),5),'dB',
      '| SSIM delta:',round(comparison.delta_ssim.mean(),5))
print('PSNR win rate:',round(100*(comparison.delta_psnr>0).mean(),2),'%')
print('PASS: Independent Test comparison complete')

## Section 15 — Save aligned visual comparisons and training curves
Each image shows **Low-Light | Model B | Model D | Ground Truth** for the *same* official Test filename; numerical scores are included in the title. The notebook saves the best- and worst-improved examples to avoid cherry-picking.

In [ ]:
VIS_DIR=RUN_ROOT/'comparison_images';VIS_DIR.mkdir(parents=True,exist_ok=True)
# Choose five best and five worst by PSNR change, unique filenames.
chosen=pd.concat([comparison.nlargest(5,'delta_psnr'),comparison.nsmallest(5,'delta_psnr')]).drop_duplicates('name')
lookup={row[0]:(row[1],row[2]) for row in test_pairs}
for _,record in chosen.iterrows():
    name=record['name'];low_path,gt_path=lookup[name]
    sources=[low_path,OUTPUTS_B/name,OUTPUTS_D/name,gt_path]
    titles=['Low-Light','Model B','Model D','Ground Truth']
    fig,axs=plt.subplots(1,4,figsize=(16,4))
    for ax,path,title in zip(axs,sources,titles):
        with Image.open(path) as im:ax.imshow(im.convert('RGB'))
        ax.set_title(title);ax.axis('off')
    fig.suptitle(f"{name} | B PSNR {record['psnr_B']:.3f}  D PSNR {record['psnr_D']:.3f}  Δ {record['delta_psnr']:+.3f} dB",fontsize=11)
    fig.tight_layout();fig.savefig(VIS_DIR/f'{Path(name).stem}_comparison.png',dpi=150,bbox_inches='tight')
    plt.close(fig)

hist=pd.read_csv(HISTORY_PATH)
fig,ax=plt.subplots(figsize=(8,4));ax.plot(hist.epoch,hist.val_psnr,label='Model D validation PSNR')
ax.axhline(val_baseline.psnr.mean(),linestyle='--',label='Model B baseline')
ax.set(xlabel='Epoch',ylabel='PSNR (dB)',title='Validation PSNR: Model D vs frozen Model B')
ax.grid(alpha=.3);ax.legend();fig.tight_layout();fig.savefig(RUN_ROOT/'training_curve.png',dpi=160);plt.close(fig)
print('Comparison figures:',len(chosen),'| folder:',VIS_DIR)
print('PASS: aligned images and training curve saved')

## Section 16 — Experiment summary + download ZIP
**Final artifacts:** `test_summary.csv`, `test_per_image.csv`, `validation_comparison.csv`, `training_history.csv`, `training_curve.png`, `comparison_images/*.png`, two model output folders and `best_model_d_head.pth`.

*ZIP includes reports and selected visual figures, not the entire cached dataset. Model B checkpoint is intentionally not duplicated.*

In [ ]:
report={
 'research':'Model B baseline plus intensity-guided HVI color correction head',
 'reference_model_b':str(MODEL_B_CKPT),'reference_model_b_sha256':MODEL_B_SHA256,
 'repo_commit':COMMIT,'best_head_epoch':int(best['epoch']),
 'validation_model_b_psnr':float(val_baseline.psnr.mean()),
 'validation_model_d_psnr':float(val_d.psnr.mean()),
 'test_count':len(test_b),
 'test_model_b_psnr':float(test_b.psnr.mean()),'test_model_d_psnr':float(test_d.psnr.mean()),
 'test_delta_psnr':float(comparison.delta_psnr.mean()),
 'test_model_b_ssim':float(test_b.ssim.mean()),'test_model_d_ssim':float(test_d.ssim.mean()),
 'test_delta_ssim':float(comparison.delta_ssim.mean()),
 'psnr_win_rate_pct':float(100*(comparison.delta_psnr>0).mean()),
 'config':CONFIG}
(RUN_ROOT/'experiment_summary.json').write_text(json.dumps(report,indent=2))
ZIP_PATH=RUN_ROOT/'ModelD_Comparison_Report.zip'
import zipfile
with zipfile.ZipFile(ZIP_PATH,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for name in ['run_config.json','training_history.csv','validation.csv','validation_comparison.csv',
                 'test_summary.csv','test_per_image.csv','training_curve.png','experiment_summary.json']:
        p=RUN_ROOT/name
        if p.exists():z.write(p,arcname=name)
    for p in sorted(VIS_DIR.glob('*.png')):z.write(p,arcname='comparison_images/'+p.name)
print('RESULT FOLDER:',RUN_ROOT)
print('REPORT ZIP:',ZIP_PATH)
print('MODEL D HEAD:',BEST_HEAD)
print('Model B baseline vs Model D: PSNR delta =',round(report['test_delta_psnr'],4),'dB; SSIM delta =',round(report['test_delta_ssim'],5))
print('PASS: final Model D experiment reports ready')

## What to send back for thesis assessment
Please share `ModelD_Comparison_Report.zip` and a few `comparison_images/*.png`. The report contains test PSNR/SSIM, per-image improvements and validation curves. If Model D is worse, we will examine **which images and color regions** degrade rather than claim an improvement.

### Important controls
- The baseline is **the original Model B Option A checkpoint**, not a retrained replacement.
- Validation images (69) are for selecting the Model D head; official Test images are not seen in training or model selection.
- The HVI transform here is a **paper-inspired adaptation**; not the original full HVI-CIDNet architecture. No guarantee of PSNR/SSIM improvement.
- Training cache is produced from **full-image** Model B inference; the 128×128 crops train only the color head.
- HVI image normalization and color correction are differentiable, but the cache uses 8-bit PNG to match saved comparison outputs. Baseline metrics can therefore differ slightly from floating-point evaluation.